# Análisis de gastos personales

En este notebook analizamos los gastos guardados en `datos.json` con **pandas** y **matplotlib**:

1. Cargar los datos.
2. Filtrar gastos.
3. Agrupar por categoría y por mes.
4. Graficar los resultados.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Mostrar los números con formato de moneda: $35,000.00
pd.options.display.float_format = "${:,.2f}".format

COLOR_BARRAS: str = "#2a78d6"

## 1. Carga de datos

Leemos `datos.json` en un DataFrame, convertimos la fecha a tipo fecha, el monto a `float` y agregamos una columna `mes` (AAAA-MM) para poder agrupar.

In [ ]:
gastos: pd.DataFrame = pd.read_json("datos.json")
gastos["fecha"] = pd.to_datetime(gastos["fecha"], format="%Y-%m-%d")
gastos["monto"] = gastos["monto"].astype(float)
gastos["mes"] = gastos["fecha"].dt.strftime("%Y-%m")

print(f"Cantidad de gastos: {len(gastos)}")
print(f"Desde {gastos['fecha'].min():%Y-%m-%d} hasta {gastos['fecha'].max():%Y-%m-%d}")
gastos.head()

## 2. Filtros

Filtramos con una condición entre corchetes: `gastos[condición]`.

In [ ]:
# Gastos de la categoría Comida
comida: pd.DataFrame = gastos[gastos["categoria"] == "Comida"]
print(f"Gastos de Comida: {len(comida)} por un total de ${comida['monto'].sum():,.2f}")
comida

In [ ]:
# Gastos de un mes en particular
mes_elegido: str = "2026-09"
gastos_del_mes: pd.DataFrame = gastos[gastos["mes"] == mes_elegido]
print(f"Gastos de {mes_elegido}: {len(gastos_del_mes)} por un total de ${gastos_del_mes['monto'].sum():,.2f}")
gastos_del_mes

In [ ]:
# Gastos mayores a $20,000, del más alto al más bajo
gastos_altos: pd.DataFrame = gastos[gastos["monto"] > 20000].sort_values("monto", ascending=False)
gastos_altos

## 3. Agrupación

### Por categoría

In [ ]:
por_categoria: pd.DataFrame = gastos.groupby("categoria")["monto"].agg(["sum", "count", "mean"])
por_categoria.columns = ["total", "cantidad", "promedio"]
por_categoria = por_categoria.sort_values("total", ascending=False)
por_categoria

### Por mes

In [ ]:
por_mes: pd.DataFrame = gastos.groupby("mes")["monto"].agg(["sum", "count", "mean"])
por_mes.columns = ["total", "cantidad", "promedio"]
por_mes

### Por categoría y mes

Tabla cruzada: cada fila es una categoría y cada columna un mes.

In [ ]:
categoria_y_mes: pd.DataFrame = gastos.groupby(["categoria", "mes"])["monto"].sum().unstack(fill_value=0)
categoria_y_mes

## 4. Gráficos

### Gráfico 1: total por categoría

In [ ]:
totales_categoria: pd.Series = por_categoria["total"].sort_values()

figura, ejes = plt.subplots(figsize=(9, 5))
barras = ejes.barh(totales_categoria.index, totales_categoria.values, color=COLOR_BARRAS, height=0.6)

for barra in barras:
    ancho: float = barra.get_width()
    ejes.text(ancho, barra.get_y() + barra.get_height() / 2, f" ${ancho:,.2f}",
              va="center", fontsize=9, color="#52514e")

ejes.set_title("Total gastado por categoría")
ejes.set_xlabel("Monto total ($)")
ejes.set_ylabel("Categoría")
ejes.xaxis.set_major_formatter(lambda valor, posicion: f"${valor:,.0f}")
ejes.set_xlim(0, totales_categoria.max() * 1.18)
ejes.grid(axis="x", color="#e0e0e0")
ejes.set_axisbelow(True)
ejes.spines["top"].set_visible(False)
ejes.spines["right"].set_visible(False)
figura.tight_layout()
plt.show()

### Gráfico 2: total por mes

In [ ]:
totales_mes: pd.Series = por_mes["total"]

figura, ejes = plt.subplots(figsize=(8, 5))
barras = ejes.bar(totales_mes.index, totales_mes.values, color=COLOR_BARRAS, width=0.5)

for barra in barras:
    altura: float = barra.get_height()
    ejes.text(barra.get_x() + barra.get_width() / 2, altura, f"${altura:,.2f}",
              ha="center", va="bottom", fontsize=9, color="#52514e")

ejes.set_title("Total gastado por mes")
ejes.set_xlabel("Mes")
ejes.set_ylabel("Monto total ($)")
ejes.yaxis.set_major_formatter(lambda valor, posicion: f"${valor:,.0f}")
ejes.grid(axis="y", color="#e0e0e0")
ejes.set_axisbelow(True)
ejes.spines["top"].set_visible(False)
ejes.spines["right"].set_visible(False)
figura.tight_layout()
plt.show()

## 5. Resumen

In [ ]:
categoria_mayor: str = por_categoria["total"].idxmax()
mes_mayor: str = por_mes["total"].idxmax()

print(f"Total gastado:             ${gastos['monto'].sum():,.2f}")
print(f"Promedio por gasto:        ${gastos['monto'].mean():,.2f}")
print(f"Categoría con mayor gasto: {categoria_mayor} (${por_categoria.loc[categoria_mayor, 'total']:,.2f})")
print(f"Mes con mayor gasto:       {mes_mayor} (${por_mes.loc[mes_mayor, 'total']:,.2f})")